In [ ]:
!pip install -U scikit-learn transformers datasets kaggle --quiet


In [ ]:
from google.colab import files
files.upload()  # Upload your kaggle.json here


In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = "/content"

# Move and set permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and extract dataset
!kaggle datasets download -d gambleryu/biggest-sql-injection-dataset -p /content
!unzip -q /content/biggest-sql-injection-dataset.zip -d /content/data
!ls /content/data


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/data/clean_sql_dataset.csv')
df = df[['Query', 'Label']].dropna()

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Query'], df['Label'], test_size=0.2, stratify=df['Label'], random_state=42
)


In [ ]:
import os

data_folder = '/content/data'
csv_files = [f for f in os.listdir(data_folder) if f.endswith('.csv')]
print("Found CSV:", csv_files)

df = pd.read_csv(os.path.join(data_folder, csv_files[0]))


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class SQLiDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        tok = self.tokenizer(self.texts[idx], padding='max_length', truncation=True,
                             max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids': tok['input_ids'].squeeze(),
            'attention_mask': tok['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = SQLiDataset(train_texts, train_labels, tokenizer)
test_ds = SQLiDataset(test_texts, test_labels, tokenizer)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)


In [ ]:
import torch.nn as nn
from transformers import BertModel

class BERT_CNN_GRU(nn.Module):
    def __init__(self, hidden_size=128, num_classes=2):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.conv = nn.Conv1d(768, 256, kernel_size=3, padding=1)
        self.gru = nn.GRU(256, hidden_size, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            out = self.bert(input_ids, attention_mask).last_hidden_state
        x = self.conv(out.permute(0, 2, 1)).permute(0, 2, 1)
        _, h = self.gru(x)
        h = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.dropout(h))


In [ ]:
import torch
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BERT_CNN_GRU().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# Early stopping
best_loss = np.inf
patience = 3
counter = 0
max_epochs = 20

history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
best_scores = []
best_labels = []

for ep in range(max_epochs):
    print(f"\nEpoch {ep+1}/{max_epochs}")
    model.train()
    correct, total, train_loss = 0, 0, 0

    for b in train_loader:
        ids, masks, labels = [b[k].to(device) for k in b]
        logits = model(ids, masks)
        loss = criterion(logits, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        train_loss += loss.item()
        correct += (torch.argmax(logits, 1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    train_loss /= len(train_loader)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)

    # Validation
    model.eval()
    correct, total, test_loss = 0, 0, 0
    all_labels, all_scores = [], []

    with torch.no_grad():
        for b in test_loader:
            ids, masks, labels = [b[k].to(device) for k in b]
            logits = model(ids, masks)
            loss = criterion(logits, labels); test_loss += loss.item()

            pred = torch.argmax(logits, 1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)

            all_labels += labels.tolist()
            all_scores += torch.softmax(logits, 1)[:, 1].tolist()

    test_acc = correct / total
    test_loss /= len(test_loader)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {test_loss:.4f}, Acc: {test_acc:.4f}")

    if test_loss < best_loss:
        best_loss = test_loss
        counter = 0
        best_scores = all_scores
        best_labels = all_labels
        torch.save(model.state_dict(), "best_model.pt")
        print("Model saved.")
    else:
        counter += 1
        print(f"Early stop counter: {counter}/{patience}")
        if counter >= patience:
            print("Early stopping triggered.")
            break

# Final Evaluation Report
print("\n Classification Report:")
pred_labels = [1 if s > 0.5 else 0 for s in best_scores]
print(classification_report(best_labels, pred_labels, target_names=["Benign", "SQLi"]))

# Accuracy & Loss Curves
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['test_loss'], label='Test Loss')
plt.title("BERT+CNN+GRU- Loss Curve"); plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.legend()

plt.subplot(1,2,2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['test_acc'], label='Test Acc')
plt.title("BERT+CNN+GRU-Accuracy Curve"); plt.xlabel("Epochs"); plt.ylabel("Accuracy"); plt.legend()
plt.tight_layout(); plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(best_labels, best_scores)
auc = roc_auc_score(best_labels, best_scores)
plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve"); plt.legend(); plt.show()

# Confusion Matrix
cm = confusion_matrix(best_labels, pred_labels)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign','SQLi'], yticklabels=['Benign','SQLi'])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix"); plt.show()


In [ ]:
from transformers import BertTokenizer
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import files


# Load dataset (replace filename if needed)
df = pd.read_csv("/content/data/clean_sql_dataset.csv")
df = df[['Query', 'Label']].dropna()
df.columns = ['Query', 'Label']

# Split dataset
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Query'], df['Label'], test_size=0.2, stratify=df['Label'], random_state=42)

# Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 64

class LSTMDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        tokens = tokenizer(self.texts[idx], padding='max_length', truncation=True,
                           max_length=MAX_LEN, return_tensors='pt')
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = LSTMDataset(train_texts, train_labels)
test_dataset = LSTMDataset(test_texts, test_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128)


In [ ]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, num_classes=2):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        _, (h_n, _) = self.lstm(x)
        h = torch.cat((h_n[-2], h_n[-1]), dim=1)  # Concatenate forward and backward hidden states
        h = self.dropout(h)
        return self.fc(h)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

device = 'cuda' if torch.cuda.is_available() else 'cpu'
VOCAB_SIZE = tokenizer.vocab_size
model = LSTMClassifier(vocab_size=VOCAB_SIZE).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loss_history, train_acc_history = [], []

for epoch in range(10):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in train_loader:
        ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        logits = model(ids)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = torch.argmax(logits, dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    train_loss_history.append(total_loss / len(train_loader))
    train_acc_history.append(acc)
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}, Accuracy = {acc:.4f}")

# Evaluate on test data
model.eval()
all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        logits = model(ids)
        probs = torch.softmax(logits, dim=1)[:, 1]
        pred = torch.argmax(logits, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(pred.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

#Metrics
print(classification_report(all_labels, all_preds, target_names=["Benign", "SQLi"]))

#ROC Curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc = roc_auc_score(all_labels, all_probs)
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve - LSTM"); plt.legend(); plt.show()

#Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Benign", "SQLi"], yticklabels=["Benign", "SQLi"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - LSTM")
plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
import torch
from torch.utils.data import Dataset, DataLoader

# Load and clean dataset
df = pd.read_csv("/content/data/clean_sql_dataset.csv")
df = df[['Query', 'Label']].dropna()
df.columns = ['Query', 'Label']

# Train-test split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Query'], df['Label'], test_size=0.2, stratify=df['Label'], random_state=42)

# BERT tokenizer for consistency with other models
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 64

class CNNDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        tokens = tokenizer(self.texts[idx], padding='max_length', truncation=True,
                           max_length=MAX_LEN, return_tensors='pt')
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = CNNDataset(train_texts, train_labels)
test_dataset = CNNDataset(test_texts, test_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128)


In [ ]:
import torch.nn as nn

class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_classes=2):
        super(CNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(embed_dim, 128, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, input_ids):
        x = self.embedding(input_ids)               # (batch, seq_len, embed_dim)
        x = x.permute(0, 2, 1)                      # (batch, embed_dim, seq_len)
        x = torch.relu(self.conv1(x))              # (batch, 128, seq_len)
        x = self.pool(x).squeeze(-1)               # (batch, 128)
        x = self.dropout(x)
        return self.fc(x)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

device = 'cuda' if torch.cuda.is_available() else 'cpu'
VOCAB_SIZE = tokenizer.vocab_size
model = CNNClassifier(vocab_size=VOCAB_SIZE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in train_loader:
        ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        logits = model(ids)

        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = torch.argmax(logits, dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}, Accuracy = {correct/total:.4f}")


In [ ]:
model.eval()
all_preds, all_labels, all_scores = [], [], []

with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        logits = model(ids)

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_scores.extend(probs.cpu().numpy())

# Classification Report
print(classification_report(all_labels, all_preds, target_names=["Benign", "SQLi"]))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Benign", "SQLi"], yticklabels=["Benign", "SQLi"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - CNN")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(all_labels, all_scores)
auc = roc_auc_score(all_labels, all_scores)
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], '--')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve - CNN"); plt.legend(); plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load and rename columns if necessary
df = pd.read_csv("/content/data/clean_sql_dataset.csv")
df = df[['Query', 'Label']].dropna()
df.columns = ['Query', 'Label']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df['Query'], df['Label'], test_size=0.2, stratify=df['Label'], random_state=42)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text into TF-IDF features
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize and train
dt_model = DecisionTreeClassifier(max_depth=20, random_state=42)
dt_model.fit(X_train_tfidf, y_train)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

# Predict
y_pred = dt_model.predict(X_test_tfidf)
y_proba = dt_model.predict_proba(X_test_tfidf)[:, 1]

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Benign", "SQLi"]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Benign", "SQLi"], yticklabels=["Benign", "SQLi"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - DT")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0,1], [0,1], '--')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve - DT"); plt.legend(); plt.show()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text to TF-IDF vectors
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Create and train KNN classifier
knn_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)  # You can tune n_neighbors
knn_model.fit(X_train_tfidf, y_train)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

# Predict
y_pred = knn_model.predict(X_test_tfidf)
y_proba = knn_model.predict_proba(X_test_tfidf)[:, 1]

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Benign", "SQLi"]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Benign", "SQLi"], yticklabels=["Benign", "SQLi"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - KNN")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve - KNN"); plt.legend(); plt.show()


In [ ]:
"""
Leakage-free 5-fold cross-validation for SQL injection detection
using BERT + CNN + GRU.

"""

# ============================================================
# 0. Install dependencies (Google Colab)
# ============================================================
!pip install -U scikit-learn transformers kaggle pandas matplotlib --quiet

# ============================================================
# 1. Kaggle access and dataset download
# ============================================================
from google.colab import files
uploaded = files.upload()  # Upload kaggle.json

import os
import json
import time
import random
import shutil
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ['KAGGLE_CONFIG_DIR'] = '/content'
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d gambleryu/biggest-sql-injection-dataset -p /content --force
!rm -rf /content/data
!mkdir -p /content/data
!unzip -q -o /content/biggest-sql-injection-dataset.zip -d /content/data
!ls -lah /content/data

# ============================================================
# 2. Reproducibility
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertModel
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    matthews_corrcoef,
)

if torch.cuda.is_available():
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# ============================================================
# 3. Configuration aligned with manuscript
# ============================================================
DATA_PATH = '/content/data/clean_sql_dataset.csv'
OUTPUT_DIR = '/content/cv_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
EPOCHS = 3
CNN_FILTERS = 128
CNN_KERNEL_SIZE = 5
GRU_HIDDEN_SIZE = 128
DROPOUT = 0.3
N_SPLITS = 5
NUM_WORKERS = 2

CONFIG = {
    'seed': SEED,
    'dataset': 'gambleryu/biggest-sql-injection-dataset',
    'data_file': DATA_PATH,
    'model_name': MODEL_NAME,
    'max_sequence_length': MAX_LEN,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'epochs_per_fold': EPOCHS,
    'cnn_filters': CNN_FILTERS,
    'cnn_kernel_size': CNN_KERNEL_SIZE,
    'gru_hidden_size': GRU_HIDDEN_SIZE,
    'dropout': DROPOUT,
    'n_splits': N_SPLITS,
    'cv_scope': '80% development partition only',
    'independent_test_usage': 'created and isolated, never evaluated in this script',
}

with open(os.path.join(OUTPUT_DIR, 'cv_run_config.json'), 'w') as f:
    json.dump(CONFIG, f, indent=2)

# ============================================================
# 4. Load dataset
#    IMPORTANT: no duplicate removal, to preserve original N.
# ============================================================
df = pd.read_csv(DATA_PATH)
df = df[['Query', 'Label']].dropna().copy()
df['Query'] = df['Query'].astype(str)

# Normalize labels only if needed.
if not pd.api.types.is_numeric_dtype(df['Label']):
    label_map = {
        'benign': 0,
        'normal': 0,
        'safe': 0,
        '0': 0,
        'sqli': 1,
        'sql injection': 1,
        'malicious': 1,
        'attack': 1,
        '1': 1,
    }
    df['Label'] = (
        df['Label']
        .astype(str)
        .str.strip()
        .str.lower()
        .map(label_map)
    )

df = df.dropna(subset=['Label']).copy()
df['Label'] = df['Label'].astype(int)

if not set(df['Label'].unique()).issubset({0, 1}):
    raise ValueError(f'Labels must be binary 0/1. Found: {sorted(df.Label.unique())}')

# Preserve original row IDs so leakage assertions are meaningful.
df = df.reset_index(drop=False).rename(columns={'index': 'original_id'})

print('\nDataset size:', len(df))
print(df['Label'].value_counts().sort_index())

# ============================================================
# 5. Create 80/20 split ONCE
#    20% independent test is isolated and NEVER used below.
# ============================================================
dev_df, heldout_test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df['Label'],
    random_state=SEED,
)

dev_df = dev_df.reset_index(drop=True)
heldout_test_df = heldout_test_df.reset_index(drop=True)

print('\nFixed split:')
print('Development (80%):', len(dev_df))
print('Held-out test (20%):', len(heldout_test_df))

# Verify no overlap.
dev_ids = set(dev_df['original_id'].tolist())
test_ids = set(heldout_test_df['original_id'].tolist())
assert dev_ids.isdisjoint(test_ids), 'LEAKAGE ERROR: development/test overlap detected.'
assert len(dev_ids) + len(test_ids) == len(df), 'Split sizes do not sum to full dataset.'

split_summary = pd.DataFrame([
    {
        'partition': 'development_80_percent',
        'n_samples': len(dev_df),
        'n_benign': int((dev_df['Label'] == 0).sum()),
        'n_sqli': int((dev_df['Label'] == 1).sum()),
    },
    {
        'partition': 'heldout_test_20_percent_not_used',
        'n_samples': len(heldout_test_df),
        'n_benign': int((heldout_test_df['Label'] == 0).sum()),
        'n_sqli': int((heldout_test_df['Label'] == 1).sum()),
    },
])
split_summary.to_csv(os.path.join(OUTPUT_DIR, 'split_summary.csv'), index=False)
print('\n', split_summary)

# ============================================================
# 6. Tokenizer and dataset class
# ============================================================
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

class SQLiDataset(Dataset):
    def __init__(self, frame, tokenizer, max_len=128):
        self.texts = frame['Query'].astype(str).tolist()
        self.labels = frame['Label'].astype(int).tolist()
        self.row_ids = frame['original_id'].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
            'original_id': torch.tensor(self.row_ids[idx], dtype=torch.long),
        }

# ============================================================
# 7. BERT + CNN + BiGRU model
#    BERT is fine-tuned (NO torch.no_grad() in forward).
# ============================================================
class BERT_CNN_GRU(nn.Module):
    def __init__(
        self,
        model_name=MODEL_NAME,
        cnn_filters=CNN_FILTERS,
        kernel_size=CNN_KERNEL_SIZE,
        gru_hidden=GRU_HIDDEN_SIZE,
        dropout=DROPOUT,
        num_classes=2,
    ):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        bert_dim = self.bert.config.hidden_size

        self.conv = nn.Conv1d(
            in_channels=bert_dim,
            out_channels=cnn_filters,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
        )
        self.relu = nn.ReLU()
        self.gru = nn.GRU(
            input_size=cnn_filters,
            hidden_size=gru_hidden,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(gru_hidden * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state

        x = x.permute(0, 2, 1)       # [B, 768, L]
        x = self.relu(self.conv(x))   # [B, F, L]
        x = x.permute(0, 2, 1)       # [B, L, F]

        _, h = self.gru(x)
        h = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(self.dropout(h))

# ============================================================
# 8. Helper functions
# ============================================================
def compute_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    fpr_value = fp / (fp + tn) if (fp + tn) else 0.0
    fnr_value = fn / (fn + tp) if (fn + tp) else 0.0

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'auc': roc_auc_score(y_true, y_prob),
        'fpr': fpr_value,
        'fnr': fnr_value,
        'mcc': matthews_corrcoef(y_true, y_pred),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_true, all_pred, all_prob, all_ids = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)

            all_true.extend(labels.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
            all_prob.extend(probs.cpu().numpy())
            all_ids.extend(batch['original_id'].cpu().numpy())

    return (
        total_loss / max(1, len(loader)),
        np.array(all_true),
        np.array(all_pred),
        np.array(all_prob),
        np.array(all_ids),
    )


def save_confusion_matrix(cm, title, path):
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_xticks([0, 1], labels=['Benign', 'SQLi'])
    ax.set_yticks([0, 1], labels=['Benign', 'SQLi'])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center')

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


def save_roc_curve(y_true, y_prob, title, path):
    fpr_curve, tpr_curve, _ = roc_curve(y_true, y_prob)
    auc_value = roc_auc_score(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr_curve, tpr_curve, label=f'AUC = {auc_value:.4f}')
    ax.plot([0, 1], [0, 1], linestyle='--')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title)
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# ============================================================
# 9. Five-fold CV ONLY on the 80% development partition
# ============================================================
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

fold_metrics = []
fold_histories = []
oof_rows = []
fold_sizes = []

X_dev = dev_df['Query']
y_dev = dev_df['Label']

for fold, (train_idx, val_idx) in enumerate(skf.split(X_dev, y_dev), start=1):
    print('\n' + '=' * 72)
    print(f'FOLD {fold}/{N_SPLITS}')
    print('=' * 72)

    fold_train_df = dev_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = dev_df.iloc[val_idx].reset_index(drop=True)

    # Strong leakage checks.
    fold_train_ids = set(fold_train_df['original_id'].tolist())
    fold_val_ids = set(fold_val_df['original_id'].tolist())

    assert fold_train_ids.isdisjoint(fold_val_ids)
    assert fold_train_ids.isdisjoint(test_ids)
    assert fold_val_ids.isdisjoint(test_ids)
    assert len(fold_train_ids) + len(fold_val_ids) == len(dev_df)

    print('Fold train size:', len(fold_train_df))
    print('Fold validation size:', len(fold_val_df))
    print('Held-out test samples used in this fold: 0')

    fold_sizes.append({
        'fold': fold,
        'train_samples': len(fold_train_df),
        'validation_samples': len(fold_val_df),
        'heldout_test_samples_used': 0,
    })

    train_ds = SQLiDataset(fold_train_df, tokenizer, MAX_LEN)
    val_ds = SQLiDataset(fold_val_df, tokenizer, MAX_LEN)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model = BERT_CNN_GRU().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    history = {
        'fold': fold,
        'epoch': [],
        'train_loss': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_f1': [],
    }

    fold_start = time.time()

    # Fixed number of epochs for every fold.
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        train_true, train_pred = [], []

        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            train_true.extend(labels.detach().cpu().numpy())
            train_pred.extend(preds.detach().cpu().numpy())

        train_loss = running_loss / max(1, len(train_loader))
        train_acc = accuracy_score(train_true, train_pred)

        val_loss, val_true, val_pred, val_prob, val_ids = evaluate(
            model, val_loader, criterion
        )
        val_acc = accuracy_score(val_true, val_pred)
        val_f1 = f1_score(val_true, val_pred, zero_division=0)

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)
        history['val_f1'].append(val_f1)

        print(
            f'Epoch {epoch}/{EPOCHS} | '
            f'Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | '
            f'Val Loss {val_loss:.4f} | Val Acc {val_acc:.4f} | Val F1 {val_f1:.4f}'
        )

    fold_seconds = time.time() - fold_start

    # Final fold metrics are computed after the fixed EPOCHS.
    val_loss, y_true, y_pred, y_prob, row_ids = evaluate(model, val_loader, criterion)
    metrics = compute_metrics(y_true, y_pred, y_prob)
    metrics.update({
        'fold': fold,
        'train_samples': len(fold_train_df),
        'validation_samples': len(fold_val_df),
        'validation_loss': val_loss,
        'training_seconds': fold_seconds,
    })
    fold_metrics.append(metrics)
    fold_histories.append(pd.DataFrame(history))

    print('\nFold metrics:')
    for key in ['accuracy', 'precision', 'recall', 'f1', 'auc', 'fpr', 'fnr', 'mcc']:
        print(f'{key}: {metrics[key]:.6f}')
    print('Confusion matrix:', metrics['tn'], metrics['fp'], metrics['fn'], metrics['tp'])

    # Out-of-fold rows for this validation fold.
    id_to_query = dev_df.set_index('original_id')['Query'].to_dict()
    for rid, yt, yp, pr in zip(row_ids, y_true, y_pred, y_prob):
        oof_rows.append({
            'fold': fold,
            'original_id': int(rid),
            'Query': id_to_query[int(rid)],
            'True_Label': int(yt),
            'Predicted_Label': int(yp),
            'SQLi_Probability': float(pr),
            'Correct': int(yt == yp),
        })

    # Per-fold confusion matrix.
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    save_confusion_matrix(
        cm,
        f'Confusion Matrix - CV Fold {fold}',
        os.path.join(OUTPUT_DIR, f'fold_{fold}_confusion_matrix.png'),
    )

    # Per-fold ROC.
    save_roc_curve(
        y_true,
        y_prob,
        f'ROC Curve - CV Fold {fold}',
        os.path.join(OUTPUT_DIR, f'fold_{fold}_roc_curve.png'),
    )

    # Per-fold loss curve.
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(history['epoch'], history['train_loss'], marker='o', label='Train loss')
    ax.plot(history['epoch'], history['val_loss'], marker='o', label='Validation loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f'Loss Curve - CV Fold {fold}')
    ax.set_xticks(history['epoch'])
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f'fold_{fold}_loss_curve.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    # Per-fold accuracy curve.
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(history['epoch'], history['train_accuracy'], marker='o', label='Train accuracy')
    ax.plot(history['epoch'], history['val_accuracy'], marker='o', label='Validation accuracy')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Accuracy Curve - CV Fold {fold}')
    ax.set_xticks(history['epoch'])
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f'fold_{fold}_accuracy_curve.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    # Release GPU memory before next fold.
    del model, optimizer, train_loader, val_loader, train_ds, val_ds
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# 10. Save fold-level outputs
# ============================================================
fold_metrics_df = pd.DataFrame(fold_metrics)
fold_metrics_df = fold_metrics_df[
    [
        'fold', 'train_samples', 'validation_samples',
        'accuracy', 'precision', 'recall', 'f1', 'auc',
        'fpr', 'fnr', 'mcc', 'tn', 'fp', 'fn', 'tp',
        'validation_loss', 'training_seconds'
    ]
]
fold_metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_fold_metrics.csv'), index=False)

pd.DataFrame(fold_sizes).to_csv(
    os.path.join(OUTPUT_DIR, 'cv_fold_sizes.csv'), index=False
)

history_df = pd.concat(fold_histories, ignore_index=True)
history_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_training_history.csv'), index=False)

oof_df = pd.DataFrame(oof_rows).sort_values('original_id').reset_index(drop=True)
oof_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_out_of_fold_predictions.csv'), index=False)

# Every development row should appear exactly once as a CV validation sample.
assert len(oof_df) == len(dev_df), 'OOF prediction count does not match development set size.'
assert oof_df['original_id'].nunique() == len(dev_df), 'A development sample appears in multiple/no folds.'
assert set(oof_df['original_id']).isdisjoint(test_ids), 'Held-out test ID leaked into OOF predictions.'

# ============================================================
# 11. Aggregate CV summary
# ============================================================
metric_cols = ['accuracy', 'precision', 'recall', 'f1', 'auc', 'fpr', 'fnr', 'mcc']
summary_rows = []

for metric in metric_cols:
    values = fold_metrics_df[metric].astype(float).values
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1))
    se = float(std / math.sqrt(N_SPLITS))

    # 95% CI using t critical for df=4 (2.776445...).
    t_crit = 2.7764451051977987
    ci_low = mean - t_crit * se
    ci_high = mean + t_crit * se

    summary_rows.append({
        'metric': metric,
        'mean': mean,
        'std': std,
        'standard_error': se,
        'ci95_low': ci_low,
        'ci95_high': ci_high,
    })

cv_summary_df = pd.DataFrame(summary_rows)
cv_summary_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_summary.csv'), index=False)

# Out-of-fold aggregate metrics across all 80% development samples.
oof_metrics = compute_metrics(
    oof_df['True_Label'].values,
    oof_df['Predicted_Label'].values,
    oof_df['SQLi_Probability'].values,
)

with open(os.path.join(OUTPUT_DIR, 'cv_oof_metrics.json'), 'w') as f:
    json.dump(oof_metrics, f, indent=2)

# Compact JSON summary for manuscript/reviewer response.
cv_json = {
    'development_samples': int(len(dev_df)),
    'heldout_test_samples_not_used': int(len(heldout_test_df)),
    'n_folds': N_SPLITS,
    'fold_train_samples': [int(x) for x in fold_metrics_df['train_samples'].tolist()],
    'fold_validation_samples': [int(x) for x in fold_metrics_df['validation_samples'].tolist()],
    'fold_accuracy': [float(x) for x in fold_metrics_df['accuracy'].tolist()],
    'mean_cv_accuracy': float(fold_metrics_df['accuracy'].mean()),
    'std_cv_accuracy': float(fold_metrics_df['accuracy'].std(ddof=1)),
    'mean_cv_f1': float(fold_metrics_df['f1'].mean()),
    'mean_cv_auc': float(fold_metrics_df['auc'].mean()),
    'oof_metrics': oof_metrics,
}
with open(os.path.join(OUTPUT_DIR, 'cv_results_summary.json'), 'w') as f:
    json.dump(cv_json, f, indent=2)

# ============================================================
# 12. Aggregate plots
# ============================================================
# 12.1 Fold accuracy
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(fold_metrics_df['fold'].astype(str), fold_metrics_df['accuracy'] * 100)
ax.axhline(
    fold_metrics_df['accuracy'].mean() * 100,
    linestyle='--',
    label=f"Mean = {fold_metrics_df['accuracy'].mean() * 100:.2f}%",
)
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Five-Fold Cross-Validation Accuracy')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'cv_fold_accuracy.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

# 12.2 Fold F1
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(fold_metrics_df['fold'].astype(str), fold_metrics_df['f1'])
ax.axhline(
    fold_metrics_df['f1'].mean(),
    linestyle='--',
    label=f"Mean = {fold_metrics_df['f1'].mean():.4f}",
)
ax.set_xlabel('Fold')
ax.set_ylabel('F1-score')
ax.set_title('Five-Fold Cross-Validation F1-score')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'cv_fold_f1.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

# 12.3 Fold AUC
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(fold_metrics_df['fold'].astype(str), fold_metrics_df['auc'])
ax.axhline(
    fold_metrics_df['auc'].mean(),
    linestyle='--',
    label=f"Mean = {fold_metrics_df['auc'].mean():.4f}",
)
ax.set_xlabel('Fold')
ax.set_ylabel('AUC')
ax.set_title('Five-Fold Cross-Validation AUC')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'cv_fold_auc.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

# 12.4 Fold-size plot -- important for reviewer response.
fold_sizes_df = pd.DataFrame(fold_sizes)
fig, ax = plt.subplots(figsize=(7, 4))
width = 0.35
x = np.arange(N_SPLITS)
ax.bar(x - width/2, fold_sizes_df['train_samples'], width, label='Fold train')
ax.bar(x + width/2, fold_sizes_df['validation_samples'], width, label='Fold validation')
ax.set_xticks(x, [str(i) for i in fold_sizes_df['fold']])
ax.set_xlabel('Fold')
ax.set_ylabel('Number of samples')
ax.set_title('Corrected CV Fold Sizes (80% Development Set Only)')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'cv_fold_sizes.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

# 12.5 OOF confusion matrix across complete 80% development partition.
oof_cm = confusion_matrix(
    oof_df['True_Label'],
    oof_df['Predicted_Label'],
    labels=[0, 1],
)
save_confusion_matrix(
    oof_cm,
    'Out-of-Fold Confusion Matrix - 80% Development Set',
    os.path.join(OUTPUT_DIR, 'cv_oof_confusion_matrix.png'),
)

# 12.6 OOF ROC across complete 80% development partition.
save_roc_curve(
    oof_df['True_Label'].values,
    oof_df['SQLi_Probability'].values,
    'Out-of-Fold ROC - 80% Development Set',
    os.path.join(OUTPUT_DIR, 'cv_oof_roc_curve.png'),
)

# 12.7 All fold ROC curves on one figure.
fig, ax = plt.subplots(figsize=(7, 6))
for fold in range(1, N_SPLITS + 1):
    fd = oof_df[oof_df['fold'] == fold]
    fpr_curve, tpr_curve, _ = roc_curve(fd['True_Label'], fd['SQLi_Probability'])
    auc_val = roc_auc_score(fd['True_Label'], fd['SQLi_Probability'])
    ax.plot(fpr_curve, tpr_curve, label=f'Fold {fold} (AUC={auc_val:.4f})')
ax.plot([0, 1], [0, 1], linestyle='--')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves Across Five Development-Set Folds')
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'cv_all_fold_roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

# 12.8 Mean epoch-wise train/validation accuracy across folds.
mean_epoch = (
    history_df
    .groupby('epoch', as_index=False)
    .agg(
        train_accuracy_mean=('train_accuracy', 'mean'),
        val_accuracy_mean=('val_accuracy', 'mean'),
        train_loss_mean=('train_loss', 'mean'),
        val_loss_mean=('val_loss', 'mean'),
    )
)
mean_epoch.to_csv(os.path.join(OUTPUT_DIR, 'cv_mean_epoch_history.csv'), index=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mean_epoch['epoch'], mean_epoch['train_accuracy_mean'], marker='o', label='Mean train accuracy')
ax.plot(mean_epoch['epoch'], mean_epoch['val_accuracy_mean'], marker='o', label='Mean validation accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Mean Accuracy Across Five CV Folds')
ax.set_xticks(mean_epoch['epoch'])
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'cv_mean_accuracy_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mean_epoch['epoch'], mean_epoch['train_loss_mean'], marker='o', label='Mean train loss')
ax.plot(mean_epoch['epoch'], mean_epoch['val_loss_mean'], marker='o', label='Mean validation loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Mean Loss Across Five CV Folds')
ax.set_xticks(mean_epoch['epoch'])
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'cv_mean_loss_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

# ============================================================
# 13. Print manuscript-ready CV summary
# ============================================================
print('\n' + '=' * 72)
print('CORRECTED FIVE-FOLD CROSS-VALIDATION SUMMARY')
print('=' * 72)
print('Full dataset:', len(df))
print('Development partition used for CV:', len(dev_df))
print('Independent test partition excluded from CV:', len(heldout_test_df))
print('\nFold sizes:')
print(fold_sizes_df.to_string(index=False))
print('\nFold metrics:')
print(fold_metrics_df.to_string(index=False))
print('\nCV summary:')
print(cv_summary_df.to_string(index=False))
print('\nOOF aggregate metrics:')
for k, v in oof_metrics.items():
    print(f'{k}: {v}')
